# Cuaderno U1-03. Etapas del proceso de modelacion

**Modelacion y Simulacion Computacional** · Maestria en Ingenieria · Universidad de Sucre

Unidad 1, Fundamentos de modelacion en ingenieria · Subtema 1.3 del plan de asignatura

Docente Daniel David Otero Meza · Periodo 2026-2

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad1/U1_03_etapas_del_proceso.ipynb)


Este cuaderno acompana la Seccion 1.3 del libro. Recorre las ocho etapas del ciclo de modelacion con sus entregables y sus criterios de salida, implementa el protocolo de diagnostico del Algoritmo 1.1 y reproduce el balance hidrico diario con el que el libro ilustra el ciclo completo sobre un lote de maiz en el departamento de Sucre.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante estara en capacidad de hacer lo siguiente.

1. Enumerar las ocho etapas del ciclo de modelacion con su entregable y su criterio de salida, segun la Tabla 1.2 del libro.
2. Aplicar el protocolo de diagnostico de los lazos de realimentacion para decidir a que etapa regresar cuando algo falla.
3. Reproducir el balance hidrico diario de la zona radicular y verificar sus cifras contra las que publica el libro.
4. Comprobar que un balance cierra de forma exacta y usar ese cierre como prueba de verificacion del programa.
5. Extender el modelo a una serie diaria de lluvia y producir el calendario de riegos de una temporada, como pide el Problema 1-26.

## Puesta a punto

La primera celda detecta el entorno e instala unicamente lo que falte. La segunda fija la semilla del curso y la paleta del libro. La tercera define las funciones de verificacion que se usan mas abajo. Ejecutelas en orden antes de continuar.

In [ ]:
# Puesta a punto del entorno. Detecta Colab e instala solo lo que falte.
import importlib
import importlib.util
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes):
    """Instala los paquetes ausentes sin reinstalar los que ya estan."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)
    return faltantes


AUSENTES = asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                     "matplotlib": "matplotlib", "sympy": "sympy"})

print("Entorno de ejecucion:", "Google Colab" if EN_COLAB else "JupyterLab local")
print("Paquetes instalados en esta sesion:", AUSENTES or "ninguno, ya estaban")

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

# Semilla unica de la asignatura. Ningun resultado depende de una corrida.
SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

# Paleta del libro. Los cuadernos usan los mismos colores que las figuras.
PALETA = {
    "azul": "#1F4E79",
    "rojo": "#B3251E",
    "verde": "#2E7D32",
    "naranja": "#E07B00",
    "gris": "#5A5A5A",
    "morado": "#6A3D9A",
}

plt.rcParams.update({
    "figure.figsize": (8.6, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
    "font.size": 10.0,
    "legend.frameon": True,
    "legend.framealpha": 0.92,
})

print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · pandas {pd.__version__}")
print(f"SymPy {sp.__version__} · semilla del curso {SEMILLA}")

In [ ]:
# Bandera de revision de los ejercicios guiados.
# Mientras valga False el cuaderno se ejecuta completo aunque falten celdas.
# Pongala en True cuando haya completado las celdas marcadas para completar.
REVISAR = False
print("REVISAR =", REVISAR)

In [ ]:
def verificar_libro(nombre, obtenido, publicado, tolerancia=1e-3, unidad=""):
    """Contrasta un resultado calculado con la cifra que publica el libro."""
    valor = float(obtenido)
    escala = abs(publicado) if publicado else 1.0
    error = abs(valor - publicado) / escala
    print(f"{nombre:<46s} calculado {valor:>12.6g} {unidad:<10s}"
          f" libro {publicado:>12.6g}  error rel. {error:.1e}")
    assert error <= tolerancia, f"{nombre} se aparta de la cifra publicada"
    return valor


def comprobar(nombre, obtenido, referencia, tolerancia=1e-3, unidad=""):
    """Revisa una celda de ejercicio contra su valor de referencia.

    Con REVISAR en False solo informa que el ejercicio sigue pendiente, de modo
    que el cuaderno nunca se detiene por una celda sin completar.
    """
    if not REVISAR:
        print(f"[pendiente]  {nombre}")
        return False
    valor = float(obtenido)
    escala = abs(referencia) if referencia else 1.0
    error = abs(valor - referencia) / escala
    marca = "correcto " if error <= tolerancia else "revisar  "
    print(f"[{marca}]  {nombre} = {valor:.6g} {unidad}"
          f"  referencia {referencia:.6g}  error rel. {error:.2e}")
    assert error <= tolerancia, f"{nombre} no coincide con la referencia"
    return True


def ruta_datos(nombre):
    """Ubica un archivo de la carpeta datos sin usar rutas absolutas.

    Funciona igual en Colab, donde el cuaderno suele abrirse en el directorio
    de trabajo, y en una copia local del repositorio, donde el cuaderno vive
    dentro de Unidad1 o de soluciones.
    """
    candidatas = (Path("datos"),
                  Path("..") / "datos",
                  Path("..") / ".." / "datos",
                  Path("03_cuadernos") / "datos")
    for base in candidatas:
        if (base / nombre).exists():
            return base / nombre
    for base in candidatas:        # la carpeta existe pero el archivo aun no
        if base.is_dir():
            return base / nombre
    return Path("datos") / nombre  # entorno nuevo, como una sesion de Colab


def cargar_o_generar(nombre, generador):
    """Lee el archivo de datos y, si no esta, lo reconstruye con la semilla."""
    ruta = ruta_datos(nombre)
    if ruta.exists():
        print(f"Datos leidos de {ruta}")
        return pd.read_csv(ruta)
    tabla = generador()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False)
    print(f"Datos regenerados con la semilla {SEMILLA} y guardados en {ruta}")
    return tabla


def integrar_trapecio(valores, muestras):
    """Regla del trapecio compatible con NumPy 1 y con NumPy 2."""
    regla = getattr(np, "trapezoid", None) or np.trapz
    return float(regla(valores, muestras))


print("Funciones auxiliares disponibles.")

## 1. Las ocho etapas y sus criterios de salida

La Tabla 1.2 del libro reune las ocho etapas del ciclo con el entregable de cada
una, el criterio que autoriza el paso a la siguiente y el error que se controla.
Las cuatro primeras construyen el modelo y ninguna produce evidencia sobre la
calidad de lo construido. Las etapas quinta, sexta y septima no construyen nada
nuevo sino que deciden si lo construido puede usarse. La octava analiza y
comunica.

In [ ]:
ETAPAS = [
    ("1 formulacion", "pregunta escrita con tolerancia y horizonte",
     "un tercero deduce que se decide", "de alcance"),
    ("2 conceptualizacion", "diagrama de frontera y registro de supuestos",
     "cada supuesto tiene prueba prevista", "estructural"),
    ("3 formulacion matematica", "ecuaciones, condiciones auxiliares y nomenclatura",
     "grados de libertad nulos y coherencia dimensional", "de formulacion"),
    ("4 implementacion", "programa con datos de prueba y control de versiones",
     "reproduce casos limite de solucion conocida", "de programacion"),
    ("5 verificacion", "estudio de refinamiento y orden observado",
     "el orden observado coincide con el teorico", "numerico"),
    ("6 calibracion", "parametros estimados con su incertidumbre",
     "parametros identificables y admisibles", "de parametros"),
    ("7 validacion", "comparacion contra datos independientes",
     "metrica dentro de la tolerancia declarada", "estructural residual"),
    ("8 comunicacion", "informe con incertidumbre y dominio de validez",
     "la pregunta inicial queda respondida", "de interpretacion"),
]
etapas = pd.DataFrame(ETAPAS, columns=["etapa", "entregable",
                                       "criterio de salida", "error controlado"])
etapas.set_index("etapa")

## 2. Los lazos de realimentacion como protocolo de diagnostico

Los lazos de la Figura 1.6 codifican una regla que ahorra semanas de trabajo.
Si la verificacion falla, el problema esta en la implementacion y no tiene
sentido tocar las hipotesis fisicas. Si la verificacion pasa y la calibracion
produce parametros sin sentido fisico, el problema esta en la formulacion
matematica, que probablemente tiene mas parametros de los que los datos
identifican. Si la calibracion es limpia y la validacion falla, el problema esta
en el modelo conceptual.

La celda siguiente escribe esa regla como una funcion, que es la forma
ejecutable del Algoritmo 1.1 del libro.

In [ ]:
def diagnosticar(verificacion_ok, parametros_con_sentido, validacion_ok):
    """Devuelve la etapa a la que hay que regresar y la razon del regreso."""
    if not verificacion_ok:
        return ("4 implementacion",
                "el programa no resuelve las ecuaciones que dice resolver, "
                "de modo que las hipotesis fisicas no estan en cuestion")
    if not parametros_con_sentido:
        return ("3 formulacion matematica",
                "la formulacion tiene mas parametros de los que los datos "
                "identifican, hay que reducir su complejidad")
    if not validacion_ok:
        return ("2 conceptualizacion",
                "las ecuaciones se resuelven bien y los parametros son "
                "admisibles, luego falta un proceso en el modelo conceptual")
    return ("8 comunicacion",
            "el modelo pasa los tres criterios y puede entregarse con su "
            "dominio de validez y su incertidumbre")


CASOS = [
    (False, True, True), (True, False, True),
    (True, True, False), (True, True, True),
]
diagnosticos = pd.DataFrame(
    [{"verificacion": v, "parametros con sentido": p, "validacion": d,
      "regresar a": diagnosticar(v, p, d)[0]} for v, p, d in CASOS])
for verificacion, parametros, validacion in CASOS:
    etapa, razon = diagnosticar(verificacion, parametros, validacion)
    print(f"verificacion {str(verificacion):<5s} parametros {str(parametros):<5s} "
          f"validacion {str(validacion):<5s} -> {etapa}")
    print(f"    {razon}\n")
diagnosticos

## 3. El ciclo completo sobre un problema de riego

El libro recorre el ciclo sobre un lote de maiz de 1 ha en el departamento de
Sucre, regado por aspersion sobre un suelo franco arenoso, con capacidad de
campo 0.24, punto de marchitez permanente 0.11 y profundidad radicular efectiva
de 0.60 m. La evapotranspiracion de referencia es de 5.2 mm por dia, el
coeficiente de cultivo vale 1.05 y la fraccion de agotamiento admisible es 0.55.

### Etapa 1, formulacion

La pregunta es en que dia debe aplicarse el proximo riego y con que lamina, con
un horizonte de un mes, resolucion diaria y tolerancia de un dia. De esa
pregunta se deduce el resto del modelo.

### Etapa 2, conceptualizacion

El modelo conceptual es un deposito con una entrada, una salida y un rebose. La
frontera es la zona radicular efectiva, los 0.60 m superiores del perfil. Lo
que sale por debajo de esa frontera es percolacion profunda y ya no pertenece al
sistema.

### Etapa 3, formulacion matematica

La Ecuacion 1.3 del libro es el balance diario de agotamiento de la zona
radicular. El agua total disponible y el umbral de riego se calculan primero,
porque son los que fijan la escala del problema.

In [ ]:
CAPACIDAD_CAMPO = 0.24        # m3/m3
MARCHITEZ = 0.11              # m3/m3
PROFUNDIDAD = 0.60            # m
ET_REFERENCIA = 5.2           # mm/dia
COEFICIENTE_CULTIVO = 1.05    # adimensional
FRACCION_ADMISIBLE = 0.55     # adimensional
EFICIENCIA_APLICACION = 0.75  # adimensional
AGOTAMIENTO_INICIAL = 12.0    # mm

AGUA_TOTAL = 1000.0 * (CAPACIDAD_CAMPO - MARCHITEZ) * PROFUNDIDAD   # mm
UMBRAL = FRACCION_ADMISIBLE * AGUA_TOTAL                            # mm
ET_CULTIVO = COEFICIENTE_CULTIVO * ET_REFERENCIA                    # mm/dia

verificar_libro("agua total disponible TAW", AGUA_TOTAL, 78.0, 1e-6, "mm")
verificar_libro("umbral de riego RAW", UMBRAL, 42.9, 1e-6, "mm")
print(f"\nevapotranspiracion del cultivo · {ET_CULTIVO:.2f} mm/dia")

### Etapa 4, implementacion

El Listado 1.2 del libro recorre el balance dia a dia. La percolacion se activa
solo cuando el balance arrojaria un agotamiento negativo, que es la conmutacion
del rebose.

In [ ]:
def programar_riego(agotamiento_inicial, etc_diaria, lluvia, umbral, dias=30):
    """Dia del proximo riego, agotamiento alcanzado y percolacion, en mm."""
    agotamiento, percolacion = agotamiento_inicial, 0.0
    for dia in range(1, dias + 1):
        balance = agotamiento + etc_diaria - lluvia.get(dia, 0.0)
        percolacion += max(0.0, -balance)
        agotamiento = max(0.0, balance)
        if agotamiento > umbral:
            return dia, agotamiento, percolacion
    return None, agotamiento, percolacion


dia_riego, lamina_neta, percolacion = programar_riego(
    AGOTAMIENTO_INICIAL, ET_CULTIVO, {3: 18.0}, UMBRAL)
lamina_bruta = lamina_neta / EFICIENCIA_APLICACION           # mm
volumen_hectarea = 10.0 * lamina_bruta                       # m3/ha
dia_sin_lluvia = programar_riego(AGOTAMIENTO_INICIAL, ET_CULTIVO, {}, UMBRAL)[0]

verificar_libro("dia del riego con la lluvia del tercer dia", dia_riego, 9, 1e-9)
verificar_libro("agotamiento alcanzado", lamina_neta, 43.14, 1e-4, "mm")
verificar_libro("lamina bruta con eficiencia 0.75", lamina_bruta, 57.52, 1e-4, "mm")
verificar_libro("volumen por hectarea", volumen_hectarea, 575.2, 1e-4, "m3/ha")
verificar_libro("dia del riego sin la lluvia", dia_sin_lluvia, 6, 1e-9)

### Etapa 5, verificacion

El libro insiste en que la verificacion es un asunto puramente matematico. Aqui
la prueba disponible es el cierre del balance, que debe ser exacto. El
agotamiento acumulado tras nueve dias es nueve veces la evapotranspiracion
diaria menos los 18 mm de lluvia, esto es 31.14 mm, que coincide con la
diferencia entre el agotamiento final y el inicial. Un cierre distinto de cero
delataria un error de programacion antes que un fenomeno fisico.

In [ ]:
acumulado = dia_riego * ET_CULTIVO - 18.0                     # mm
diferencia = lamina_neta - AGOTAMIENTO_INICIAL                # mm
residuo = acumulado - diferencia + percolacion                # mm

verificar_libro("agotamiento acumulado en nueve dias", acumulado, 31.14, 1e-4, "mm")
verificar_libro("diferencia entre agotamiento final e inicial", diferencia, 31.14, 1e-4, "mm")
print(f"\nresiduo del cierre · {residuo:.3e} mm")
assert abs(residuo) < 1e-9, "el balance de agua debe cerrar de forma exacta"
print("El balance cierra en el orden del redondeo de la maquina, de modo que la "
      "etapa 5 queda superada y la etapa 6 puede empezar.")

### La trayectoria del agotamiento

La figura muestra el estado del deposito dia a dia, con el umbral de riego y el
efecto de la lluvia del tercer dia. Sin esa lluvia el riego habria correspondido
al sexto dia.

In [ ]:
def trayectoria(agotamiento_inicial, etc_diaria, lluvia, dias=12):
    """Serie diaria del agotamiento de la zona radicular, en mm."""
    agotamiento, serie = agotamiento_inicial, [agotamiento_inicial]
    for dia in range(1, dias + 1):
        agotamiento = max(0.0, agotamiento + etc_diaria - lluvia.get(dia, 0.0))
        serie.append(agotamiento)
    return np.array(serie)


dias = np.arange(0, 13)
con_lluvia = trayectoria(AGOTAMIENTO_INICIAL, ET_CULTIVO, {3: 18.0})
sin_lluvia = trayectoria(AGOTAMIENTO_INICIAL, ET_CULTIVO, {})

figura, eje = plt.subplots()
eje.step(dias, con_lluvia, where="post", color=PALETA["azul"], lw=1.9,
         label="Con lluvia efectiva de 18 mm el tercer dia")
eje.step(dias, sin_lluvia, where="post", color=PALETA["rojo"], lw=1.7, ls="--",
         label="Sin lluvia")
eje.axhline(UMBRAL, color=PALETA["naranja"], lw=1.4,
            label=f"Umbral de riego RAW = {UMBRAL:.1f} mm")
eje.axhline(AGUA_TOTAL, color=PALETA["gris"], lw=1.0, ls=(0, (1, 2)),
            label=f"Agua total disponible TAW = {AGUA_TOTAL:.1f} mm")
eje.plot([dia_riego], [lamina_neta], "o", color=PALETA["azul"], ms=7, zorder=5)
eje.plot([dia_sin_lluvia], [sin_lluvia[dia_sin_lluvia]], "o",
         color=PALETA["rojo"], ms=7, zorder=5)
eje.set_xlabel("Dia desde el inicio del balance")
eje.set_ylabel("Agotamiento de la zona radicular (mm)")
eje.set_xlim(0, 12)
eje.legend(loc="upper left", fontsize=8.8)
eje.set_title("Balance hidrico diario de la Seccion 1.3 del libro")
plt.show()

print(f"Riego el dia {dia_riego} con {lamina_neta:.2f} mm netos, esto es "
      f"{lamina_bruta:.2f} mm brutos o {volumen_hectarea:.1f} m3 por hectarea.")
print(f"Sin la lluvia el riego habria correspondido al dia {dia_sin_lluvia}.")

### Etapas 6, 7 y 8

La calibracion de este modelo se reduce a ajustar el coeficiente de cultivo,
porque el resto de los parametros son medibles de forma independiente. La
validacion exigiria contenidos de humedad medidos con sonda, tomados en fechas
que no participaran del ajuste. La comunicacion debe declarar una limitacion que
el calculo no hace evidente, y es que la lluvia entro como dato observado y no
como prediccion, lo cual restringe el uso del resultado a la programacion de muy
corto plazo.

## 4. Ejercicios guiados

Cinco celdas incompletas con la marca `# COMPLETE:`, un valor de partida
deliberadamente incorrecto y su verificacion inmediata.

### Ejercicio 1. Agua disponible y umbral en otro suelo

Un segundo lote, de suelo franco arcilloso, tiene capacidad de campo 0.32, punto
de marchitez permanente 0.19, profundidad radicular efectiva de 0.45 m y
fraccion de agotamiento admisible de 0.50.

In [ ]:
# COMPLETE: escriba la funcion que devuelve el agua total disponible en mm, que
# es 1000 por la diferencia de contenidos volumetricos por la profundidad
# radicular en metros, y el umbral de riego, que es la fraccion admisible por el
# agua total disponible.
def agua_disponible(capacidad_campo, marchitez, profundidad, fraccion):
    return float("nan"), float("nan")   # valores de partida incorrectos


taw_lote_dos, raw_lote_dos = agua_disponible(0.32, 0.19, 0.45, 0.50)

In [ ]:
uno = comprobar("TAW del segundo lote", taw_lote_dos, 58.5, 1e-6, "mm")
dos = comprobar("RAW del segundo lote", raw_lote_dos, 29.25, 1e-6, "mm")

### Ejercicio 2. El dia del riego en el segundo lote

El segundo lote arranca con un agotamiento de 8 mm, tiene una evapotranspiracion
de cultivo de 4.8 mm por dia y recibe 12 mm de lluvia efectiva el cuarto dia.

In [ ]:
# COMPLETE: use programar_riego con los datos del segundo lote y deje el dia del
# riego en dia_lote_dos y el agotamiento alcanzado, en mm, en neto_lote_dos.
dia_lote_dos = 0                # valor de partida deliberadamente incorrecto
neto_lote_dos = 0.0             # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("dia del riego en el segundo lote", dia_lote_dos, 7, 1e-9)
dos = comprobar("agotamiento en el segundo lote", neto_lote_dos, 29.6, 1e-6, "mm")

### Ejercicio 3. La lamina bruta y el volumen aplicado

La eficiencia de aplicacion del segundo lote, que riega por surcos, es de 0.60.

In [ ]:
# COMPLETE: escriba la funcion que convierte la lamina neta en lamina bruta,
# dividiendo por la eficiencia de aplicacion, y calcule con ella la lamina bruta
# del segundo lote y el volumen que exige una hectarea, en metros cubicos. Una
# lamina de 1 mm sobre 1 ha equivale a 10 m3.
def lamina_bruta_y_volumen(lamina_neta, eficiencia):
    return float("nan"), float("nan")   # valores de partida incorrectos


bruta_lote_dos, volumen_lote_dos = lamina_bruta_y_volumen(neto_lote_dos, 0.60)

In [ ]:
uno = comprobar("lamina bruta del segundo lote", bruta_lote_dos, 49.3333, 1e-4, "mm")
dos = comprobar("volumen del segundo lote", volumen_lote_dos, 493.333, 1e-4, "m3/ha")

### Ejercicio 4. El protocolo de diagnostico

Tres situaciones de un proyecto real. En la primera, el estudio de refinamiento
muestra un orden observado de 0.5 cuando el esquema es de orden dos. En la
segunda, el refinamiento es impecable pero la calibracion arroja una
conductividad hidraulica negativa. En la tercera, todo lo anterior esta en orden
y la metrica sobre los datos reservados triplica la tolerancia declarada.

In [ ]:
# COMPLETE: llame a diagnosticar con los tres argumentos que describe cada
# situacion y deje en la lista etapas_respuesta el nombre de la etapa a la que
# hay que regresar en cada caso, en el mismo orden del enunciado.
etapas_respuesta = ["8 comunicacion", "8 comunicacion", "8 comunicacion"]

In [ ]:
ESPERADAS = ["4 implementacion", "3 formulacion matematica", "2 conceptualizacion"]
aciertos = sum(a == b for a, b in zip(etapas_respuesta, ESPERADAS))
for i, (dada, esperada) in enumerate(zip(etapas_respuesta, ESPERADAS), start=1):
    print(f"  situacion {i} · respondio {dada:<26s}"
          f"{'' if dada == esperada else 'revisar'}")
comprobar("situaciones acertadas, sobre 3", aciertos, 3, 1e-9)

### Ejercicio 5. Problema 1-26, el calendario de una temporada

El Problema 1-26 pide ampliar el modelo del Listado 1.2 para que acepte una
serie diaria de lluvia y devuelva el calendario de riegos de una temporada
completa.

La serie de 120 dias que se usa aqui se lee de la carpeta `datos` y, si no
estuviera disponible, se reconstruye con la semilla del curso. Es sintetica y
representa una temporada de lluvias del Caribe seco colombiano, con probabilidad
diaria variable y laminas de distribucion gamma. La regla de operacion es que,
cuando el agotamiento supera el umbral, se riega hasta llevar el deposito a
capacidad de campo, esto es hasta un agotamiento nulo.

In [ ]:
def generar_lluvia_temporada():
    """Serie diaria sintetica de lluvia efectiva de una temporada, en mm."""
    generador = np.random.default_rng(SEMILLA)
    dias_serie = np.arange(1, 121)
    probabilidad = 0.30 + 0.22 * np.sin(np.pi * (dias_serie - 15) / 120)
    llueve = generador.random(120) < probabilidad
    laminas = generador.gamma(1.6, 9.0, 120)
    return pd.DataFrame({"dia": dias_serie,
                         "lluvia_efectiva_mm": np.round(
                             np.where(llueve, laminas, 0.0), 1)})


temporada = cargar_o_generar("U1_lluvia_diaria_sucre.csv", generar_lluvia_temporada)
serie_lluvia = temporada["lluvia_efectiva_mm"].to_numpy(dtype=float)

print(f"dias con lluvia · {int((serie_lluvia > 0).sum())} de {len(serie_lluvia)}")
print(f"lluvia acumulada · {serie_lluvia.sum():.1f} mm")
print(f"lamina diaria maxima · {serie_lluvia.max():.1f} mm")
temporada.head(8)

In [ ]:
# COMPLETE: escriba la funcion que recorre la serie diaria de lluvia, aplica el
# mismo balance del Listado 1.2 y devuelve la lista de riegos como tuplas de
# dia, lamina neta y lamina bruta, junto con la percolacion acumulada. Despues
# de cada riego el agotamiento vuelve a cero. Aplicola a la serie de la
# temporada con el agotamiento inicial, la evapotranspiracion y el umbral del
# lote del libro, con eficiencia de 0.75.
def calendario_de_riegos(serie, etc_diaria, umbral,
                         agotamiento_inicial=12.0, eficiencia=0.75):
    return [], float("nan")   # valores de partida deliberadamente incorrectos


riegos, percolacion_temporada = calendario_de_riegos(
    serie_lluvia, ET_CULTIVO, UMBRAL)
numero_riegos = len(riegos)
bruto_temporada = float(sum(r[2] for r in riegos)) if riegos else 0.0

In [ ]:
uno = comprobar("numero de riegos de la temporada", numero_riegos, 5, 1e-9)
dos = comprobar("lamina bruta acumulada", bruto_temporada, 300.9067, 1e-4, "mm")
tres = comprobar("percolacion acumulada", percolacion_temporada, 202.84, 1e-4, "mm")
if REVISAR and uno:
    figura, eje = plt.subplots(figsize=(8.6, 3.6))
    eje.bar(temporada["dia"], serie_lluvia, color=PALETA["azul"], width=0.9,
            label="Lluvia efectiva diaria")
    for i, (dia_r, _, _) in enumerate(riegos):
        eje.axvline(dia_r, color=PALETA["rojo"], lw=1.2, ls="--",
                    label="Riego" if i == 0 else None)
    eje.set_xlabel("Dia de la temporada")
    eje.set_ylabel("Lamina de lluvia efectiva (mm)")
    eje.legend(loc="upper right")
    eje.set_title("Problema 1-26, calendario de riegos de una temporada")
    plt.show()

## 5. Problemas del capitulo

El Problema 1-6 pide clasificar el modelo de agotamiento de la zona radicular
segun los seis criterios y justificar cada posicion, y se resuelve en el
Ejercicio 4 del cuaderno U1-02. El Problema 1-26 queda resuelto en el Ejercicio 5
de este cuaderno.

Queda por discutir, en la celda siguiente, una limitacion que el calculo no hace
evidente. La lluvia entro como dato observado y no como prediccion, de modo que
el calendario de la temporada completa no es un pronostico sino una
reconstruccion de lo que habria convenido hacer. Explique que cambiaria en las
etapas 1, 7 y 8 del ciclo si el modelo tuviera que operar hacia adelante con un
pronostico de lluvia de tres dias.

### Respuesta del estudiante

*Escriba aqui. Un parrafo por cada una de las tres etapas, indicando que
entregable cambia y que criterio de salida se vuelve mas exigente.*

## Cierre

### Lista de comprobacion

Marque cada punto solo si puede hacerlo sin mirar el cuaderno.

- Nombrar las ocho etapas del ciclo con su entregable y su criterio de salida sin consultar la Tabla 1.2.
- Decidir a que etapa regresar cuando la verificacion, la calibracion o la validacion fallan.
- Programar un balance diario que cierre de forma exacta y usar ese cierre como prueba de verificacion.
- Distinguir calibrar de validar y explicar por que usar los mismos datos para ambas cosas invalida la conclusion.
- Extender un modelo a una serie observada y producir un calendario de operacion.

### Que revisar si algo no salio

- Si el dia del riego no coincide, revise que la comparacion sea estricta, esto es que el riego se dispare cuando el agotamiento supera el umbral y no cuando lo iguala.
- Si el cierre del balance no da cero, compruebe que la percolacion se acumula antes de recortar el agotamiento a cero, porque el orden de esas dos operaciones cambia el resultado.
- Si el calendario de la temporada difiere, confirme que el agotamiento vuelve a cero despues de cada riego y que la serie de lluvia se leyo de `datos` o se regenero con la semilla 20262.
- Para la teoria, relea la Seccion 1.3 del libro, la Tabla 1.2, la Figura 1.6 y el Algoritmo 1.1.

### Declaracion del uso de asistentes de programacion

Si empleo un asistente basado en modelos de lenguaje para resolver alguna celda, declarelo en la entrega, indique en cual y describa que prueba aplico para convencerse de que el codigo es correcto. La regla de la asignatura es que el estudiante responde por el resultado que firma, con independencia de quien escriba las lineas.